# 15 — Ensemble transformerów (uśrednianie prawdopodobieństw)

Uśrednienie prawdopodobieństw HerBERT-base + HerBERT-large-LoRA + XLM-R-base (checkpointy z 05) + wspólne strojenie progów na val. Bez treningu.

Modele liczone w chmurze dołączane przez `data/results/external_probas/{nazwa}_proba_{val,test}.npy`.

In [1]:
import glob, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
from scipy.special import expit
from sklearn.metrics import f1_score, jaccard_score, hamming_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification
warnings.filterwarnings("ignore")
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RANDOM_STATE=42
PROCESSED_DIR=Path("../data/processed"); RESULTS_DIR=Path("../data/results"); TR=Path("../data/transformers")
device="cuda" if torch.cuda.is_available() else "cpu"
tw_val=pd.read_csv(PROCESSED_DIR/"twitteremo_val.csv"); tw_test=pd.read_csv(PROCESSED_DIR/"twitteremo_test.csv")
for d in (tw_val,tw_test): d["tekst"]=d["tekst"].fillna("")
y_val,y_test=tw_val[EMOTIONS].values,tw_test[EMOTIONS].values

def ckpt(name):
    c=sorted(glob.glob(str(TR/name/"checkpoint-*")))
    return c[-1] if c else None

MODELS={"herbert-base-cased":"allegro/herbert-base-cased",
        "xlm-roberta-base":"FacebookAI/xlm-roberta-base",
        "herbert-large-cased-lora":"allegro/herbert-large-cased"}
print({k:ckpt(k) for k in MODELS})

/path/to/repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'herbert-base-cased': '../data/transformers/herbert-base-cased/checkpoint-14344', 'xlm-roberta-base': '../data/transformers/xlm-roberta-base/checkpoint-14344', 'herbert-large-cased-lora': '../data/transformers/herbert-large-cased-lora/checkpoint-10758'}


In [ ]:
def opt_thr(yt,yp):
    thr=np.full(len(EMOTIONS),0.5)
    for i in range(len(EMOTIONS)):
        bf,bt=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            f=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[i]=bt
    return thr
def f1_ci(yt,yp,n=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed); m=len(yt); base=f1_score(yt,yp,average="macro",zero_division=0)
    b=[f1_score(yt[i],yp[i],average="macro",zero_division=0) for i in (rng.integers(0,m,m) for _ in range(n))]
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi

@torch.no_grad()
def predict(model,tok,texts,bs=32):
    # Autocast fp16 — TA SAMA precyzja co w notatniku 05 (TrainingArguments(fp16=True)
    # włącza autocast również w Trainer.predict). Bez tego prawdopodobieństwa różnią się
    # o tysięczne, co przy poszarpanej krzywej F1(próg) przestawia progi wybrane na val
    # i rozjeżdża składowe z tabelą główną nawet o 0,011 (herbert-large-lora 0,552 vs 0,563).
    out=[]
    for i in range(0,len(texts),bs):
        enc=tok(list(texts[i:i+bs]),truncation=True,max_length=128,padding=True,return_tensors="pt").to(device)
        with torch.autocast(device,dtype=torch.float16,enabled=(device=="cuda")):
            out.append(expit(model(**enc).logits.float().cpu().numpy()))
    return np.vstack(out)

def load_model(name,base):
    cp=ckpt(name); tok=AutoTokenizer.from_pretrained(base)
    if "lora" in name:
        from peft import PeftModel
        m=AutoModelForSequenceClassification.from_pretrained(base,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
        m=PeftModel.from_pretrained(m,cp)
    else:
        m=AutoModelForSequenceClassification.from_pretrained(cp)
    return m.to(device).eval(),tok

In [3]:
proba_val={}; proba_test={}; singles=[]
for name,base in MODELS.items():
    try:
        m,tok=load_model(name,base)
        pv=predict(m,tok,tw_val["tekst"].tolist()); pt=predict(m,tok,tw_test["tekst"].tolist())
        proba_val[name]=pv; proba_test[name]=pt
        thr=opt_thr(y_val,pv); f1=f1_score(y_test,(pt>=thr).astype(int),average="macro",zero_division=0)
        singles.append({"model":name,"f1_macro":round(f1,3)})
        print(f"  {name}: F1-Macro={f1:.3f}")
        del m; torch.cuda.empty_cache()
    except Exception as e:
        print(f"  {name}: SKIP ({type(e).__name__}: {e})")
print("modeli w ensemble:",len(proba_val))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  15%|█▍        | 30/201 [00:00<00:00, 297.62it/s]

Loading weights:  30%|██▉       | 60/201 [00:00<00:00, 198.94it/s]

Loading weights:  41%|████      | 82/201 [00:00<00:00, 179.74it/s]

Loading weights:  53%|█████▎    | 107/201 [00:00<00:00, 197.48it/s]

...

  xlm-roberta-base: F1-Macro=0.534


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 25894.44it/s]


BertForSequenceClassification LOAD REPORT from: allegro/herbert-large-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly 

  herbert-large-cased-lora: F1-Macro=0.552
modeli w ensemble: 3


In [4]:
# Zewnętrzne modele (Kaggle/Colab): pary {nazwa}_proba_val.npy + {nazwa}_proba_test.npy
# w data/results/external_probas/ (np. bielik45, herbert_large_full). Val jest konieczny —
# progi ensemble stroimy na val; sam *_proba_test.npy nie wystarczy.
EXT = RESULTS_DIR / "external_probas"
if EXT.exists():
    for fv in sorted(EXT.glob("*_proba_val.npy")):
        name = fv.name.replace("_proba_val.npy", "")
        ft = EXT / f"{name}_proba_test.npy"
        if not ft.exists():
            print(f"  {name}: brak {ft.name} — pomijam"); continue
        pv, pt = np.load(fv), np.load(ft)
        assert pv.shape == (len(y_val), len(EMOTIONS)) and pt.shape == (len(y_test), len(EMOTIONS)), \
            f"{name}: zly ksztalt (val {pv.shape}, test {pt.shape})"
        proba_val[name] = pv; proba_test[name] = pt
        thr = opt_thr(y_val, pv)
        f1 = f1_score(y_test, (pt >= thr).astype(int), average="macro", zero_division=0)
        singles.append({"model": name + " (ext)", "f1_macro": round(f1, 3)})
        print(f"  {name} (ext): F1-Macro={f1:.3f}")
else:
    print("brak data/results/external_probas/ — ensemble tylko z lokalnych checkpointow")
print("modeli w ensemble:", len(proba_val))

  bielik45 (ext): F1-Macro=0.593


  herbert_large_full (ext): F1-Macro=0.582


  xlmr_large_full (ext): F1-Macro=0.571
modeli w ensemble: 6


In [5]:
# Ensemble: średnia prawdopodobieństw + progi na val
P_val=np.mean(list(proba_val.values()),axis=0); P_test=np.mean(list(proba_test.values()),axis=0)
thr=opt_thr(y_val,P_val); pred=(P_test>=thr).astype(int)
base,lo,hi=f1_ci(y_test,pred)
row={"model":f"ensemble ({len(proba_val)})","f1_macro":round(base,3),"ci_low":round(lo,3),"ci_high":round(hi,3),
     "f1_micro":round(f1_score(y_test,pred,average="micro",zero_division=0),3),
     "jaccard_macro":round(jaccard_score(y_test,pred,average="macro",zero_division=0),3),
     "hamming_loss":round(hamming_loss(y_test,pred),3)}
res=pd.DataFrame(singles+[row]); res.to_csv(RESULTS_DIR/"ensemble_transformers.csv",index=False)
print(f"\nENSEMBLE F1-Macro={base:.3f} [{lo:.3f}, {hi:.3f}]")
display(res)
best_single=max(s["f1_macro"] for s in singles)
(RESULTS_DIR/"ensemble_summary.md").write_text(
  f"# Ensemble transformerów\n\n- Najlepszy pojedynczy: {best_single:.3f}\n- Ensemble: {base:.3f} [{lo:.3f}, {hi:.3f}] (Δ={base-best_single:+.3f})\n")
print(f"Δ vs najlepszy pojedynczy ({best_single:.3f}): {base-best_single:+.3f}")


ENSEMBLE F1-Macro=0.602 [0.561, 0.637]


,model,f1_macro,ci_low,ci_high,f1_micro,jaccard_macro,hamming_loss
0,herbert-base-cased,0.550,NaN,NaN,NaN,NaN,NaN
1,xlm-roberta-base,0.534,NaN,NaN,NaN,NaN,NaN
2,herbert-large-cased-lora,0.552,NaN,NaN,NaN,NaN,NaN
3,bielik45 (ext),0.593,NaN,NaN,NaN,NaN,NaN
4,herbert_large_full (ext),0.582,NaN,NaN,NaN,NaN,NaN
5,xlmr_large_full (ext),0.571,NaN,NaN,NaN,NaN,NaN
6,ensemble (6),0.602,0.561,0.637,0.684,0.44,0.09


Δ vs najlepszy pojedynczy (0.593): +0.009
